# allin1 — Music Structure Analysis

Runs [mir-aidj/all-in-one](https://github.com/mir-aidj/all-in-one) on your MP3 files and produces `allin1.json` output files ready to drop into the `web-app/public/analysis/<slug>/` folder of the TimeCues Studio repo.

**GPU is optional** — the model runs on CPU too, just slower (~3–5 min/song on CPU vs ~30–60s on T4 GPU).  
To enable GPU: Runtime → Change runtime type → T4 GPU.

In [ ]:
import subprocess, torch

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    print('No GPU — running on CPU (slower but works fine)')

print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')

## Step 1 — Install dependencies

Run this cell once per session. Takes ~2 minutes.

In [ ]:
# ── System deps ───────────────────────────────────────────────────────────────
!apt-get install -y -q ffmpeg

# ── Python deps ───────────────────────────────────────────────────────────────
# allin1 + natten (GPU wheel auto-selected from shi-labs index)
!pip install -q allin1 natten -f https://shi-labs.com/natten/wheels/

# madmom from git — supports NumPy 2.x (PyPI stable release does not)
!pip install -q git+https://github.com/CPJKU/madmom.git

print('\nAll dependencies installed.')

## Step 2 — Load your MP3 files

Choose one option:

- **`"drive"`** — mount Google Drive and point to a folder containing your MP3s
- **`"upload"`** — upload MP3 files directly from your computer

Set `LOAD_METHOD` below, then run the corresponding cell.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
LOAD_METHOD = "drive"   # "drive" or "upload"

# If LOAD_METHOD == "drive": path inside your Google Drive to a flat folder of MP3s
# Each MP3 should be the original song file (same files as in songs/*/*.mp3 in the repo)
DRIVE_MP3_FOLDER = "MyDrive/timecues/mp3s"

In [ ]:
# ── Load files ────────────────────────────────────────────────────────────────
from pathlib import Path

mp3_paths = []

if LOAD_METHOD == "drive":
    from google.colab import drive
    drive.mount('/content/drive')
    folder = Path('/content/drive') / DRIVE_MP3_FOLDER
    mp3_paths = sorted(folder.glob('**/*.mp3'))
    print(f'Found {len(mp3_paths)} MP3 file(s) in {folder}')

elif LOAD_METHOD == "upload":
    from google.colab import files
    upload_dir = Path('/content/uploads')
    upload_dir.mkdir(exist_ok=True)
    print('Select your MP3 files to upload:')
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = upload_dir / name
        dest.write_bytes(data)
        mp3_paths.append(dest)
    print(f'\nUploaded {len(mp3_paths)} file(s)')

for p in mp3_paths:
    print(' ', p.name)

## Step 3 — Run analysis

Each song takes ~30–60 seconds on a T4 GPU. Results are saved to `/content/results/<slug>/allin1.json`.

In [ ]:
import allin1
import json
import re
import time
from pathlib import Path

# ── natten compatibility shim (handles natten >= 0.17 API changes) ────────────
def _apply_natten_shim():
    try:
        from natten.functional import natten1dav  # noqa — already present
        return False
    except ImportError:
        pass
    import torch
    import natten.functional as _nf

    def natten1dqkrpb(query, key, rpb, kernel_size, dilation=1):
        B, H, L, D = query.shape
        r = kernel_size // 2
        offsets = torch.arange(-r, r + 1, device=query.device) * dilation
        positions = torch.arange(L, device=query.device)
        src_idx = (positions.unsqueeze(1) + offsets.unsqueeze(0)).clamp(0, L - 1)
        key_nbrs = key[:, :, src_idx, :]
        scores = (query.unsqueeze(3) * key_nbrs).sum(-1)
        rpb_sel = rpb[:, torch.arange(kernel_size, device=query.device)]
        return scores + rpb_sel.unsqueeze(0).unsqueeze(2)

    def natten1dav(attn, value, kernel_size, dilation=1):
        _, _, L, _ = attn.shape
        r = kernel_size // 2
        offsets = torch.arange(-r, r + 1, device=attn.device) * dilation
        positions = torch.arange(L, device=attn.device)
        src_idx = (positions.unsqueeze(1) + offsets.unsqueeze(0)).clamp(0, L - 1)
        v_nbrs = value[:, :, src_idx, :]
        return (attn.unsqueeze(-1) * v_nbrs).sum(-2)

    def natten2dqkrpb(query, key, rpb, kernel_size, dilation=1):
        B, H, h, w, D = query.shape
        r = kernel_size // 2
        oy = torch.arange(-r, r + 1, device=query.device) * dilation
        ox = torch.arange(-r, r + 1, device=query.device) * dilation
        sy = (torch.arange(h, device=query.device).unsqueeze(1) + oy.unsqueeze(0)).clamp(0, h - 1)
        sx = (torch.arange(w, device=query.device).unsqueeze(1) + ox.unsqueeze(0)).clamp(0, w - 1)
        key_nbrs = key[:, :, sy.unsqueeze(2).expand(-1, kernel_size),
                            sx.unsqueeze(0).expand(h, -1, -1), :]
        scores = (query.unsqueeze(3).unsqueeze(4) * key_nbrs).sum(-1)
        return scores + rpb.unsqueeze(0).unsqueeze(2).unsqueeze(2)

    def natten2dav(attn, value, kernel_size, dilation=1):
        _, _, h, w, _, _ = attn.shape
        r = kernel_size // 2
        oy = torch.arange(-r, r + 1, device=attn.device) * dilation
        ox = torch.arange(-r, r + 1, device=attn.device) * dilation
        sy = (torch.arange(h, device=attn.device).unsqueeze(1) + oy.unsqueeze(0)).clamp(0, h - 1)
        sx = (torch.arange(w, device=attn.device).unsqueeze(1) + ox.unsqueeze(0)).clamp(0, w - 1)
        v_nbrs = value[:, :, sy.unsqueeze(2).expand(-1, kernel_size),
                             sx.unsqueeze(0).expand(h, -1, -1), :]
        return (attn.unsqueeze(-1) * v_nbrs).sum((-3, -2))

    _nf.natten1dqkrpb = natten1dqkrpb
    _nf.natten1dav    = natten1dav
    _nf.natten2dqkrpb = natten2dqkrpb
    _nf.natten2dav    = natten2dav
    return True

_shimmed = _apply_natten_shim()
if _shimmed:
    print('natten compatibility shim applied')

# ── Helpers ───────────────────────────────────────────────────────────────────
def slugify(name):
    name = re.sub(r'\.[^.]+$', '', name)
    name = name.lower()
    name = re.sub(r'[^a-z0-9]+', '-', name)
    return name.strip('-')

def label_to_type(label):
    label = label.lower().strip()
    mapping = {
        'intro': 'intro', 'verse': 'verse', 'pre-chorus': 'buildup',
        'chorus': 'drop', 'bridge': 'breakdown', 'break': 'breakdown',
        'instrumental': 'verse', 'outro': 'outro', 'solo': 'verse',
        'interlude': 'breakdown',
    }
    return mapping.get(label, label)

# ── Run analysis ─────────────────────────────────────────────────────────────
OUT_DIR = Path('/content/results')
OUT_DIR.mkdir(exist_ok=True)

errors = []

for mp3 in mp3_paths:
    print(f'\n━━━ {mp3.name} ━━━')
    t0 = time.time()
    try:
        result = allin1.analyze(str(mp3))
        elapsed = time.time() - t0

        slug = slugify(mp3.name)

        sections = []
        raw_boundaries = []
        for seg in result.segments:
            sections.append({
                'time':    round(seg.start, 4),
                'endTime': round(seg.end, 4),
                'type':    label_to_type(seg.label),
                'label':   seg.label.capitalize(),
            })
            raw_boundaries.append(round(seg.start, 4))
        if sections:
            raw_boundaries.append(round(result.segments[-1].end, 4))

        duration = result.segments[-1].end if result.segments else 0.0

        output = {
            'algorithm': 'allin1',
            'algoName':  'All-In-One (allin1)',
            'audioFile': mp3.name,
            'duration':  round(duration, 4),
            'bpm':       round(result.bpm, 4),
            'beatPositions':     [round(t, 4) for t in result.beat_positions],
            'downbeatPositions': [round(t, 4) for t in result.downbeat_positions],
            'sections':          sections,
            'rawBoundaries':     raw_boundaries,
            'computedAt':        int(time.time()),
            'elapsedSec':        round(elapsed, 2),
        }

        slug_dir = OUT_DIR / slug
        slug_dir.mkdir(exist_ok=True)
        out_path = slug_dir / 'allin1.json'
        out_path.write_text(json.dumps(output, indent=2))

        print(f'BPM: {result.bpm:.2f} | Segments: {len(sections)} | {elapsed:.1f}s')
        print(f'Saved → {out_path}')

    except Exception as e:
        print(f'ERROR: {e}')
        errors.append((mp3.name, str(e)))

print(f'\n━━━ Done: {len(mp3_paths) - len(errors)}/{len(mp3_paths)} succeeded ━━━')
if errors:
    print('Failed:')
    for name, err in errors:
        print(f'  {name}: {err}')

## Step 4 — Download results

Downloads a zip file containing all `<slug>/allin1.json` files.

In the repo, extract and copy each folder into `web-app/public/analysis/`:
```
web-app/public/analysis/<slug>/allin1.json
```

In [ ]:
import shutil
from google.colab import files

zip_path = '/content/allin1_results'
shutil.make_archive(zip_path, 'zip', '/content/results')

print('Downloading allin1_results.zip ...')
files.download(zip_path + '.zip')